In [ ]:
# ============================================================
# CONFIGURATION:
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from collections import defaultdict
from numpy.random import default_rng
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt

# --- INPUT CONFIG ---
PROJECT_ROOT = Path("../../../Image_Authenticity_prediction/").resolve()
NUMBER_EXPECTED_FILES = 25
REMOVED_IMAGES_FILE = PROJECT_ROOT / "Dataset" / "AIGCIQA2023" / "removed_images.txt"
SINGLE_SCORES_FILES = PROJECT_ROOT / "Dataset" / "Single_scores" 

# --- OUTPUT CONFIG ---
BASE_DIR = PROJECT_ROOT / "main" / "Experiments"/ "Outputs"/ "Analisys_Noise_Celing"
PLOT_DIR = BASE_DIR / "Plots"
SAVE_PER_IMAGE_CSV = BASE_DIR / "per_image_means_QAM.csv"
RAW_OUT_CSV = BASE_DIR / "images_values_gt5_raw.csv"
MEANS_OUT_CSV = BASE_DIR / "images_means_gt5.csv"
OUT_CSV = BASE_DIR / "split_half_reliability_authenticity.csv"
SAVE_PER_SUBJECT_CORR_CSV = BASE_DIR / "per_subject_QA_bin_corr.csv"

# --- ANALYSIS CONFIG ---
BINS = 20
N_SPLITS = 20
RANDOM_SEED = 12345
QUALITY_AUTHENTICITY_BINS = np.linspace(0, 5, 21)  # 20 bins
SOURCE_CSVS = sorted(SINGLE_SCORES_FILES.glob("*_scores.csv"))  # 01_scores.csv ... 25_scores.csv from Single_scores folder

# --- OTHER CONFIG ---
USE_STRICT_COMPLETE_CASE = True  # set True to enforce all subjects rate all images
EXPECT_N_FILES = 25

# Create output directories
BASE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Output directories created/verified:")
print(f"  - BASE_DIR: {BASE_DIR}")
print(f"  - PLOT_DIR: {PLOT_DIR}")

In [ ]:
# Load removed IDs
removed_ids = set()

if REMOVED_IMAGES_FILE.exists():
    print(f"Loading removed IDs from: {REMOVED_IMAGES_FILE}\n")

    with open(REMOVED_IMAGES_FILE, "r", encoding="utf-8") as f:
        for line in f:
            orig = line.rstrip("\n")
            line = orig.strip()
            if not line:
                continue

            file_path = Path(line)
            stem = file_path.stem       # e.g. "67" from "67.png"

            try:
                stem_int = int(stem)
            except ValueError:
                print(f"[WARN] Could not parse integer from stem '{stem}' (line: {orig})")
                continue

            removed_ids.add(stem_int)

    print("Final removed_ids set (ints):")
    print(removed_ids)
else:
    print(f"[WARN] removed_images file not found at {REMOVED_IMAGES_FILE}, no IDs will be excluded.")


# --- VERIFY SOURCE FILES ---
print(f"\n=== Loading {NUMBER_EXPECTED_FILES} Single Scores Files ===")
print(f"Source directory: {SINGLE_SCORES_FILES}")
print(f"Found {len(SOURCE_CSVS)} CSV files matching '*_scores.csv'")
if len(SOURCE_CSVS) != NUMBER_EXPECTED_FILES:
    print(f"[WARN] Expected {NUMBER_EXPECTED_FILES} files, found {len(SOURCE_CSVS)}")
else:
    print(f"✓ Confirmed: {len(SOURCE_CSVS)} source files loaded")


# --- CONTAINERS ---
quality = defaultdict(list)       # uniqueID (col B) -> [Q values]
authenticity = defaultdict(list)  # uniqueID (col B) -> [A values]
match = defaultdict(list)         # uniqueID (col B) -> [M values]

loaded_files = []
total_rows = 0


def _coerce_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def load_file(file_path: Path) -> pd.DataFrame:
    """
    Standardize to columns:
    ['uniqueID','Quality','Authenticity','Match']
    where uniqueID = column B; Q/A/M = columns E/F/G.
    Uses POSITIONAL columns so it's robust to header names.
    """
    
    # Read all, then index by position safely
    try:
        raw_data = pd.read_csv(file_path, header=0)
    except Exception:
        raw_data = pd.read_csv(file_path, header=None)

    # Check we have enough columns (at least 7 for B,E,F,G)
    if raw_data.shape[1] < 7:
        raise ValueError(
            f"Expected at least 7 columns to access positions B,E,F,G; got {raw_data.shape[1]}"
        )

    # B is the second column (index 1) in your example
    unique_id_column = _coerce_numeric(raw_data.iloc[:, 1]).astype("Int64")  # column B (pos 1)
    quality_column = _coerce_numeric(raw_data.iloc[:, 4])                  # column E (pos 4)
    authenticity_column = _coerce_numeric(raw_data.iloc[:, 5])                  # column F (pos 5)
    match_column = _coerce_numeric(raw_data.iloc[:, 6])                  # column G (pos 6)

    out = pd.DataFrame({
        "uniqueID": unique_id_column,
        "Quality": quality_column,
        "Authenticity": authenticity_column,
        "Match": match_column
    })

    # Drop rows with missing ID
    out = out.dropna(subset=["uniqueID"])
    # Drop rows with all-NaN metrics
    out = out.dropna(subset=["Quality", "Authenticity", "Match"], how="all")

    # Ensure uniqueID is Int64
    out["uniqueID"] = out["uniqueID"].astype("Int64")

    return out


# --- SELECT ONLY THE 25 SOURCE FILES ---
# they follow the pattern "##_scores.csv" (01_scores.csv ... 25_scores.csv):
source_csvs = sorted(SINGLE_SCORES_FILES.glob("*_scores.csv"))
source_excels = sorted(p for p in SINGLE_SCORES_FILES.iterdir() if p.suffix.lower() in {".xlsx", ".xls"})
# Prefer the explicit CSV pattern if that's your data; otherwise include Excels as well.
files = source_csvs if source_csvs else source_excels

if not files:
    raise FileNotFoundError(
        f"No source files found. Looked for '*_scores.csv' or Excel files in {SINGLE_SCORES_FILES}"
    )


# --- LOAD ALL FILES ---
for file_path in files:
    try:
        dataframe = load_file(file_path)

        for _, row in dataframe.iterrows():
            uid = row["uniqueID"]  # Int64 / integer-like
            if pd.isna(uid):
                continue
            uid = int(uid)

            # --- SKIP REMOVED IMAGES ---
            if uid in removed_ids:
                continue

            quality_val, authenticity_val, match_val = row["Quality"], row["Authenticity"], row["Match"]

            if pd.notna(quality_val):
                quality[uid].append(float(quality_val))
            if pd.notna(authenticity_val):
                authenticity[uid].append(float(authenticity_val))
            if pd.notna(match_val):
                match[uid].append(float(match_val))

        loaded_files.append(file_path.name)
        total_rows += len(dataframe)
    except Exception as e:
        print(f"[WARN] Skipping {file_path.name}: {e}")


# --- OVERALL MEANS ---
def flat_values(metric_dict):
    return [val for vals in metric_dict.values() for val in vals]


all_Q = np.array(flat_values(quality), dtype=float)
all_A = np.array(flat_values(authenticity), dtype=float)
all_M = np.array(flat_values(match), dtype=float)


def safe_mean(arr):
    return float(np.nanmean(arr)) if arr.size else float("nan")


overall_Q_mean = safe_mean(all_Q)
overall_A_mean = safe_mean(all_A)
overall_M_mean = safe_mean(all_M)

print("\n=== Overall Means ===")
print(f"Quality mean:       {overall_Q_mean:.4f} (N={all_Q.size})")
print(f"Authenticity mean:  {overall_A_mean:.4f} (N={all_A.size})")
print(f"Match mean:         {overall_M_mean:.4f} (N={all_M.size})")

# --- PER-IMAGE MEANS + COUNTS ---
all_ids = sorted(set(quality) | set(authenticity) | set(match))
rows = []
for uid in all_ids:
    quality_vals = np.array(quality.get(uid, []), dtype=float)
    authenticity_vals = np.array(authenticity.get(uid, []), dtype=float)
    match_vals = np.array(match.get(uid, []), dtype=float)
    rows.append({
        "uniqueID": uid,
        "Q_mean": safe_mean(quality_vals), "Q_n": quality_vals.size,
        "A_mean": safe_mean(authenticity_vals), "A_n": authenticity_vals.size,
        "M_mean": safe_mean(match_vals), "M_n": match_vals.size,
    })

per_image_df = pd.DataFrame(rows).sort_values("uniqueID").reset_index(drop=True)

# Preview in Jupyter
try:
    display(per_image_df.head(10))
except NameError:
    pass

# Save per-image means
BASE_DIR.mkdir(parents=True, exist_ok=True)
per_image_df.to_csv(SAVE_PER_IMAGE_CSV, index=False)
print(f"\nSaved per-image means to: {SAVE_PER_IMAGE_CSV}")
print(f"Loaded {len(loaded_files)} files (expected ~{NUMBER_EXPECTED_FILES}).")
print(f"Total usable rows (before filtering by removed IDs): {total_rows}")
print(f"Unique images (after filtering): {len(all_ids)}")

In [ ]:
def plot_hist(series, metric_name, plot_dir, bins=20):
    metric_series = series.dropna().astype(float)
    if metric_series.empty:
        print(f"[WARN] No data for {metric_name}; skipping histogram.")
        return
    # If values look like proportions (<=1), plot as percentages
    as_percent = np.nanmax(metric_series.values) <= 1.05
    if as_percent:
        data_values = metric_series.values * 100.0
        x_label = f"{metric_name} (%)"
        hist_range = (0, 100)
    else:
        data_values = metric_series.values
        x_label = f"{metric_name}"
        # auto-range with a small margin
        xmin, xmax = np.nanmin(data_values), np.nanmax(data_values)
        span = xmax - xmin
        hist_range = (xmin - 0.02*span, xmax + 0.02*span) if span > 0 else (xmin - 1, xmax + 1)

    plt.figure(figsize=(8, 5))
    plt.hist(data_values, bins=bins, range=hist_range)
    plt.title(f"Histogram of per-image mean {metric_name}")
    plt.xlabel(x_label)
    plt.ylabel("Count of images")
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()

    fname = plot_dir / f"hist_per_image_mean_{metric_name.replace(' ', '')}.png"
    plt.savefig(fname, dpi=600, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

# --- PLOT ---
PLOT_DIR.mkdir(parents=True, exist_ok=True)
plot_hist(per_image_df["Q_mean"], "Quality", PLOT_DIR, bins=BINS)
plot_hist(per_image_df["A_mean"], "Authenticity", PLOT_DIR, bins=BINS)
plot_hist(per_image_df["M_mean"], "Match", PLOT_DIR, bins=BINS)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- BUILD LONG RAW DF FROM DICTS ---
def dict_to_long(metric_dict, metric_name):
    rows = []
    for uid, vals in metric_dict.items():
        for val in vals:
            rows.append({"uniqueID": uid, "metric": metric_name, "value": float(val)})
    return pd.DataFrame(rows)

raw_q = dict_to_long(quality, "Quality")
raw_a = dict_to_long(authenticity, "Authenticity")
raw_m = dict_to_long(match, "Match")
raw_all = pd.concat([raw_q, raw_a, raw_m], ignore_index=True)

# --- FIND RAW VALUES > 5 ---
raw_gt5 = raw_all[raw_all["value"] > 5].sort_values(["metric", "value"], ascending=[True, False])

print("=== Raw values > 5 (any metric) ===")
print(f"Total rows flagged: {len(raw_gt5)}")
print(raw_gt5.head(20))  # preview
if len(raw_gt5):
    BASE_DIR.mkdir(parents=True, exist_ok=True)
    raw_gt5.to_csv(RAW_OUT_CSV, index=False)
    print(f"Saved full list to: {RAW_OUT_CSV}")

# --- ALSO CHECK PER-IMAGE MEANS > 5 (sanity) ---
means_cols = ["Q_mean", "A_mean", "M_mean"]
missing_means = [col for col in means_cols if col not in per_image_df.columns]
if missing_means:
    print(f"[WARN] per_image_df missing expected columns: {missing_means}")
else:
    means_gt5 = per_image_df[
        (per_image_df["Q_mean"] > 5) | 
        (per_image_df["A_mean"] > 5) | 
        (per_image_df["M_mean"] > 5)
    ].copy()

    print("\n=== Per-image MEANS > 5 ===")
    print(f"Images flagged: {len(means_gt5)}")
    print(means_gt5.head(20))  # preview

    if len(means_gt5):
        BASE_DIR.mkdir(parents=True, exist_ok=True)
        means_gt5.to_csv(MEANS_OUT_CSV, index=False)
        print(f"Saved per-image means >5 to: {MEANS_OUT_CSV}")


In [ ]:
# Load one file

def read_auth_from_csv(file_path: Path) -> pd.Series:
    """
    Returns a pandas Series: index = uniqueID (integer from col B),
    values = Authenticity (col F).
    Skips rows with removed IDs.
    Works whether CSV has a header or not; uses positional columns.
    """
    # Try reading with header, fall back to no header
    try:
        raw_data = pd.read_csv(file_path, header=0)
    except Exception:
        raw_data = pd.read_csv(file_path, header=None)

    if raw_data.shape[1] <= 5:
        raise ValueError(f"{file_path.name}: expected >= 6 columns to access B/F; got {raw_data.shape[1]}")

    # --- Extract columns ---
    unique_ids = pd.to_numeric(raw_data.iloc[:, 1], errors="coerce").astype("Int64")  # column B → integer ID
    auth_values = pd.to_numeric(raw_data.iloc[:, 5], errors="coerce")                 # column F

    # --- Build Series ---
    metric_series = pd.Series(auth_values.values, index=unique_ids.values, name=file_path.stem)

    # --- Drop invalid IDs ---
    metric_series = metric_series.dropna()                           # remove rows where authenticity is NA
    metric_series = metric_series[~metric_series.index.isna()]                   # remove rows where uniqueID is NA
    metric_series.index = metric_series.index.astype(int)            # convert Int64 → Python int

    # --- Skip removed IDs ---
    metric_series = metric_series[~metric_series.index.isin(removed_ids)]

    return metric_series


# ------------- BUILD WIDE MATRIX -------------
series_list = []
colnames = []
for file_path in SOURCE_CSVS:
    try:
        metric_series = read_auth_from_csv(file_path)
        # if duplicate IDs in a file, average them
        metric_series = metric_series.groupby(level=0).mean()
        series_list.append(metric_series)
        colnames.append(file_path.stem)
    except Exception as e:
        print(f"[WARN] Skipping {file_path.name}: {e}")

if not series_list:
    raise RuntimeError("No participant series loaded successfully.")

# Outer-join all series on uniqueID to build images × participants matrix
auth_mat = pd.concat(series_list, axis=1)
auth_mat.columns = colnames
print(f"Matrix shape (images × participants): {auth_mat.shape}")

# ---------------- COVERAGE AUDIT ----------------
coverage = auth_mat.notna()
n_images, n_participants = coverage.shape
per_subj_counts = coverage.sum(axis=0)  # images rated by each subject
per_img_counts  = coverage.sum(axis=1)  # subjects per image

print("\n=== Coverage audit ===")
print(f"Images: {n_images}, Participants: {n_participants}")
print("Per-subject coverage (first 10):")
print((per_subj_counts.sort_values(ascending=True).head(10)).to_string())
print("Per-image coverage (summary):")
print(per_img_counts.describe().to_string())

# Flag low-coverage subjects (e.g., < 90% images)
low_subj_threshold = int(np.ceil(0.90 * n_images))
low_subj = per_subj_counts[per_subj_counts < low_subj_threshold]
if not low_subj.empty:
    print(f"\n[WARN] {len(low_subj)} subjects rate <90% of images. Lowest 5:")
    print(low_subj.sort_values().head(5).to_string())

# Flag low-coverage images (e.g., < 90% subjects)
low_img_threshold = int(np.ceil(0.90 * n_participants))
n_low_imgs = int((per_img_counts < low_img_threshold).sum())
if n_low_imgs:
    print(f"[WARN] {n_low_imgs} images have <90% subject coverage.")

# ---------------- FILTERING POLICY ----------------
# Choose ONE of the following:

USE_STRICT_COMPLETE_CASE = True  # set True to enforce all subjects rate all images

if USE_STRICT_COMPLETE_CASE:
    # Keep only images rated by all participants (complete rows)
    auth_filt = auth_mat.dropna(axis=1, how="all")     # drop empty participants (unlikely)
    auth_filt = auth_filt.dropna(axis=0, how="any")    # strict: drop any image with a missing
    print(f"\nStrict complete-case -> Matrix: {auth_filt.shape} (images × participants)")
else:
    # Thresholded filtering
    min_images_per_subject = int(np.ceil(0.90 * n_images))        # subject must rate >=90% images
    keep_subj = per_subj_counts >= min_images_per_subject
    auth_filt = auth_mat.loc[:, keep_subj]
    n_participants_f = auth_filt.shape[1]

    min_subjects_per_image = int(np.ceil(0.90 * n_participants_f))  # image must have >=90% subjects
    keep_imgs = auth_filt.notna().sum(axis=1) >= min_subjects_per_image
    auth_filt = auth_filt.loc[keep_imgs]

    print(f"\nThresholded filtering -> Matrix: {auth_filt.shape} (images × participants)")
    print(f"Kept {keep_subj.sum()}/{n_participants} subjects; "
          f"Kept {keep_imgs.sum()}/{n_images} images")

if auth_filt.shape[0] < 3 or auth_filt.shape[1] < 2:
    raise RuntimeError("Not enough data after filtering. Loosen thresholds or inspect coverage.")

# ---------------- SPLIT-HALF with MIN RATERS PER HALF ----------------
rng = default_rng(RANDOM_SEED)
participants = np.array(auth_filt.columns)
n_participants = len(participants)

n1 = n_participants // 2
n2 = n_participants - n1
assert n1 + n2 == n_participants

# Require at least this many raters in EACH half per image to use the image
min_raters_per_half = max(2, int(np.floor(0.7 * n1)))  # e.g., 70% of each half (tweakable)

def pearson_corr(series_a: pd.Series, series_b: pd.Series) -> float:
    both = pd.concat([series_a, series_b], axis=1).dropna()
    if both.shape[0] < 3:
        return np.nan
    return float(both.iloc[:,0].corr(both.iloc[:,1], method='pearson'))

def fisher_z(correlation: float) -> float:
    correlation = np.clip(correlation, -0.999999, 0.999999)
    return np.arctanh(correlation)

def fisher_inv(z_value: float) -> float:
    return np.tanh(z_value)

def spearman_brown(correlation: float, m: float = 2.0) -> float:
    if np.isnan(correlation): return np.nan
    return (m * correlation) / (1 + (m - 1) * correlation)

records = []
for split_idx in range(N_SPLITS):
    permutation_indices = rng.permutation(n_participants)
    group1 = participants[permutation_indices[:n1]]
    group2 = participants[permutation_indices[n1:]]

    # Mean per image in each half (skipna allows missing raters inside the half)
    mean1 = auth_filt[group1].mean(axis=1, skipna=True)
    mean2 = auth_filt[group2].mean(axis=1, skipna=True)

    # Enforce that each image has sufficient raters in BOTH halves
    cnt1 = auth_filt[group1].count(axis=1)  # non-NA counts per image in half 1
    cnt2 = auth_filt[group2].count(axis=1)  # non-NA counts per image in half 2
    use_mask = (cnt1 >= min_raters_per_half) & (cnt2 >= min_raters_per_half)

    r = pearson_corr(mean1[use_mask], mean2[use_mask])
    z = fisher_z(r) if not np.isnan(r) else np.nan
    r_sb = spearman_brown(r, m=2.0)
    z_sb = fisher_z(r_sb) if not np.isnan(r_sb) else np.nan

    records.append({
        "split_idx": split_idx+1,
        "n_in_g1": int(len(group1)),
        "n_in_g2": int(len(group2)),
        "min_raters_per_half": int(min_raters_per_half),
        "n_images_used": int(use_mask.sum()),
        "r_split": r,
        "z_split": z,
        "r_sb_full": r_sb,
        "z_sb_full": z_sb
    })

splits_df = pd.DataFrame(records)

z_mean = np.nanmean(splits_df["z_split"].values)
r_mean = fisher_inv(z_mean)
z_sb_mean = np.nanmean(splits_df["z_sb_full"].values)
r_sb_mean = fisher_inv(z_sb_mean)

print("\n=== Split-half reliability (Authenticity) with missing-data controls ===")
print(f"Participants: {n_participants} (split {n1}/{n2}), splits: {N_SPLITS}")
print(f"Requirement: ≥{min_raters_per_half} raters per half per image")
print(f"Mean Pearson r (Fisher-z avg): {r_mean:.4f}")
print(f"Spearman–Brown corrected r (full 25-participant mean): {r_sb_mean:.4f}")
print(f"Images per split (used), mean ± SD: "
      f"{splits_df['n_images_used'].mean():.1f} ± {splits_df['n_images_used'].std(ddof=1):.1f}")



# ------------- SPLIT-HALF CORRELATIONS -------------
rng = default_rng(RANDOM_SEED)
participants = np.array(auth_mat.columns)
n_participants = len(participants)
if n_participants < 2:
    raise RuntimeError("Need at least 2 participants for split-half.")

n1 = n_participants // 2            # 12 (for 25 participants this is 12)
n2 = n_participants - n1            # 13
assert n1 + n2 == n_participants

def pearson_corr(series_a: pd.Series, series_b: pd.Series) -> float:
    """Pearson correlation across their common (non-NaN) images. Returns NaN if <3 paired points."""
    both = pd.concat([series_a, series_b], axis=1).dropna()
    if both.shape[0] < 3:
        return np.nan
    return float(both.iloc[:,0].corr(both.iloc[:,1], method='pearson'))

def fisher_z(correlation: float) -> float:
    correlation = np.clip(correlation, -0.999999, 0.999999)
    return np.arctanh(correlation)

def fisher_inv(z_value: float) -> float:
    return np.tanh(z_value)

def spearman_brown(correlation: float, m: float = 2.0) -> float:
    """Prophecy formula; m=2 for split-half -> full-length."""
    if np.isnan(correlation):
        return np.nan
    return (m * correlation) / (1 + (m - 1) * correlation)

records = []
for split_idx in range(N_SPLITS):
    permutation_indices = rng.permutation(n_participants)
    group1 = participants[permutation_indices[:n1]]
    group2 = participants[permutation_indices[n1:]]

    mean1 = auth_mat[group1].mean(axis=1, skipna=True)
    mean2 = auth_mat[group2].mean(axis=1, skipna=True)

    r = pearson_corr(mean1, mean2)
    z = fisher_z(r) if not np.isnan(r) else np.nan
    r_sb = spearman_brown(r, m=2.0)  # halves -> full 25-subject mean (factor ~2)
    z_sb = fisher_z(r_sb) if not np.isnan(r_sb) else np.nan

    n_common = pd.concat([mean1, mean2], axis=1).dropna().shape[0]

    records.append({
        "split_idx": split_idx+1,
        "n_in_g1": int(len(group1)),
        "n_in_g2": int(len(group2)),
        "n_images_used": int(n_common),
        "r_split": r,
        "z_split": z,
        "r_sb_full": r_sb,
        "z_sb_full": z_sb
    })

splits_df = pd.DataFrame(records)

# Fisher-z average (more stable) and back-transform
z_mean = np.nanmean(splits_df["z_split"].values)
r_mean = fisher_inv(z_mean)

z_sb_mean = np.nanmean(splits_df["z_sb_full"].values)
r_sb_mean = fisher_inv(z_sb_mean)

print("\n=== Split-half reliability (Authenticity) ===")
print(f"Participants: {n_participants} (split {n1}/{n2}), splits: {N_SPLITS}")
print(f"Mean Pearson r (Fisher-z averaged): {r_mean:.4f}")
print(f"Spearman–Brown corrected r (full 25-participant mean): {r_sb_mean:.4f}")
print(f"Images per split (paired, mean ± SD): "
      f"{splits_df['n_images_used'].mean():.1f} ± {splits_df['n_images_used'].std(ddof=1):.1f}")

# Save per-split details
BASE_DIR.mkdir(parents=True, exist_ok=True)
splits_df.to_csv(OUT_CSV, index=False)
print(f"Saved per-split stats to: {OUT_CSV}")


In [ ]:
# Positional column map (0-based indices in pandas iloc):
# B = 1 (uniqueID), E = 4 (Quality), F = 5 (Authenticity), G = 6 (Match)
METRICS = {
    "Quality": {"col_idx": 4, "outfile": BASE_DIR / "split_half_reliability_quality.csv"},
    "Authenticity": {"col_idx": 5, "outfile": BASE_DIR / "split_half_reliability_authenticity.csv"},
    "Match": {"col_idx": 6, "outfile": BASE_DIR / "split_half_reliability_match.csv"},
}

def read_metric_series_from_csv(file_path: Path, metric_col_idx: int) -> pd.Series:
    """
    Returns a Series per participant:
      index = uniqueID (integer from column B),
      values = <metric at metric_col_idx>.
    Uses positional columns so headers don't matter.
    Drops empty IDs and NaNs. Averages duplicates. Skips removed IDs.
    """
    # Load CSV with or without header
    try:
        raw_data = pd.read_csv(file_path, header=0)
    except Exception:
        raw_data = pd.read_csv(file_path, header=None)

    # Need at least columns for B (idx 1) and metric_col_idx
    if raw_data.shape[1] <= max(1, metric_col_idx):
        raise ValueError(f"{file_path.name}: expected > {metric_col_idx} columns; got {raw_data.shape[1]}")

    # Column B → uniqueID as Int64
    unique_ids = pd.to_numeric(raw_data.iloc[:, 1], errors="coerce").astype("Int64")
    # Metric column (e.g., E/F/G)
    metric_values = pd.to_numeric(raw_data.iloc[:, metric_col_idx], errors="coerce")

    # Build Series
    metric_series = pd.Series(metric_values.values, index=unique_ids.values, name=file_path.stem)

    # Drop rows with missing metric
    metric_series = metric_series.dropna()
    # Drop rows with missing ID
    metric_series = metric_series[~metric_series.index.isna()]

    # Convert index to Python int
    metric_series.index = metric_series.index.astype(int)

    # Skip removed IDs
    metric_series = metric_series[~metric_series.index.isin(removed_ids)]

    # Average duplicates within file
    metric_series = metric_series.groupby(level=0).mean()

    return metric_series


def build_matrix(metric_col_idx: int) -> pd.DataFrame:
    """Outer-join all participant series into an images × participants matrix for the chosen metric."""
    series_list, colnames = [], []
    for file_path in SOURCE_CSVS:
        try:
            metric_series = read_metric_series_from_csv(file_path, metric_col_idx)
            series_list.append(metric_series)
            colnames.append(file_path.stem)
        except Exception as e:
            print(f"[WARN] Skipping {file_path.name}: {e}")
    if not series_list:
        raise RuntimeError("No participant series loaded successfully.")
    matrix = pd.concat(series_list, axis=1)
    matrix.columns = colnames
    return matrix

def compute_split_half_by_subjects(matrix: pd.DataFrame,
                                   use_strict_complete_case: bool = True,
                                   subj_cov_thresh: float = 0.90,
                                   img_cov_thresh: float = 0.90,
                                   min_half_fraction: float = 0.70,
                                   n_splits: int = 20,
                                   seed: int = 12345) -> tuple[pd.DataFrame, dict]:
    """
    matrix: images × participants values (floats, NaNs allowed).
    Returns (splits_df, summary_dict).
    """
    # Coverage audit
    coverage = matrix.notna()
    n_images, n_participants = coverage.shape
    per_subj_counts = coverage.sum(axis=0)
    per_img_counts  = coverage.sum(axis=1)

    # Filtering policy
    if use_strict_complete_case:
        filtered_matrix = matrix.dropna(axis=1, how="all")
        filtered_matrix = filtered_matrix.dropna(axis=0, how="any")
        policy = "strict_complete_case"
    else:
        min_images_per_subject = int(np.ceil(subj_cov_thresh * n_images))
        keep_subj = per_subj_counts >= min_images_per_subject
        filtered_matrix = matrix.loc[:, keep_subj]
        n_participants_f = filtered_matrix.shape[1]
        min_subjects_per_image = int(np.ceil(img_cov_thresh * n_participants_f))
        keep_imgs = filtered_matrix.notna().sum(axis=1) >= min_subjects_per_image
        filtered_matrix = filtered_matrix.loc[keep_imgs]
        policy = f"thresholded(subj≥{subj_cov_thresh:.0%}, img≥{img_cov_thresh:.0%})"

    if filtered_matrix.shape[0] < 3 or filtered_matrix.shape[1] < 2:
        raise RuntimeError(f"Not enough data after filtering ({filtered_matrix.shape}). Adjust thresholds.")

    # Split-half with min raters/half
    rng = default_rng(seed)
    participants = np.array(filtered_matrix.columns)
    n_participants = len(participants)
    n1 = n_participants // 2
    n2 = n_participants - n1
    min_raters_per_half = max(2, int(np.floor(min_half_fraction * n1)))

    def pearson_corr(series_a: pd.Series, series_b: pd.Series) -> float:
        both = pd.concat([series_a, series_b], axis=1).dropna()
        if both.shape[0] < 3:
            return np.nan
        return float(both.iloc[:, 0].corr(both.iloc[:, 1], method='pearson'))

    def fisher_z(correlation: float) -> float:
        return np.arctanh(np.clip(correlation, -0.999999, 0.999999)) if np.isfinite(correlation) else np.nan

    def fisher_inv(z_value: float) -> float:
        return np.tanh(z_value) if np.isfinite(z_value) else np.nan

    def spearman_brown(correlation: float, m: float = 2.0) -> float:
        if not np.isfinite(correlation): return np.nan
        return (m * correlation) / (1 + (m - 1) * correlation)

    records = []
    for split_idx in range(n_splits):
        permutation_indices = rng.permutation(n_participants)
        group1 = participants[permutation_indices[:n1]]
        group2 = participants[permutation_indices[n1:]]

        mean1 = filtered_matrix[group1].mean(axis=1, skipna=True)
        mean2 = filtered_matrix[group2].mean(axis=1, skipna=True)

        cnt1 = filtered_matrix[group1].count(axis=1)  # non-NA counts per image in half 1
        cnt2 = filtered_matrix[group2].count(axis=1)  # non-NA counts per image in half 2
        use_mask = (cnt1 >= min_raters_per_half) & (cnt2 >= min_raters_per_half)

        r = pearson_corr(mean1[use_mask], mean2[use_mask])
        z = fisher_z(r)
        r_sb = spearman_brown(r, m=2.0)
        z_sb = fisher_z(r_sb)

        records.append({
            "split_idx": split_idx + 1,
            "n_in_g1": int(len(group1)),
            "n_in_g2": int(len(group2)),
            "min_raters_per_half": int(min_raters_per_half),
            "n_images_used": int(use_mask.sum()),
            "r_split": r,
            "z_split": z,
            "r_sb_full": r_sb,
            "z_sb_full": z_sb
        })

    splits_df = pd.DataFrame(records)
    z_mean = np.nanmean(splits_df["z_split"].values)
    r_mean = fisher_inv(z_mean)
    z_sb_mean = np.nanmean(splits_df["z_sb_full"].values)
    r_sb_mean = fisher_inv(z_sb_mean)

    summary = {
        "policy": policy,
        "matrix_shape_after_filter": filtered_matrix.shape,
        "participants_after_filter": n_participants,
        "half_sizes": (n1, n2),
        "min_raters_per_half": int(min_raters_per_half),
        "mean_r_split": r_mean,
        "mean_r_sb_full": r_sb_mean,
        "noise_ceiling_r_max": float(np.sqrt(r_sb_mean)) if np.isfinite(r_sb_mean) and r_sb_mean >= 0 else np.nan
    }
    return splits_df, summary

# ---------------- RUN FOR A, Q, M ----------------
results = {}
for metric_name, cfg in METRICS.items():
    print(f"\n================ {metric_name} =================")
    matrix = build_matrix(cfg["col_idx"])
    print(f"Matrix shape (images × participants): {matrix.shape}")

    splits_df, summary = compute_split_half_by_subjects(
        matrix,
        use_strict_complete_case=False,  # set True if you want only complete images
        subj_cov_thresh=0.90,            # subject must rate ≥90% of images
        img_cov_thresh=0.90,             # image must have ≥90% of kept subjects
        min_half_fraction=0.70,          # ≥70% of each half required per image
        n_splits=N_SPLITS,
        seed=RANDOM_SEED
    )

    # Print summary
    print(f"Filtering policy: {summary['policy']}")
    print(f"After filtering: {summary['matrix_shape_after_filter']} (images × participants)")
    n1, n2 = summary["half_sizes"]
    print(f"Split {n1}/{n2}, min raters/half: {summary['min_raters_per_half']}")
    print(f"Mean Pearson r (Fisher-z avg): {summary['mean_r_split']:.4f}")
    print(f"Spearman–Brown corrected r (full {summary['participants_after_filter']}-participant mean): {summary['mean_r_sb_full']:.4f}")
    print(f"Noise ceiling for Pearson r vs model (sqrt of SB): {summary['noise_ceiling_r_max']:.4f}")
    print(f"Images per split used (mean ± SD): "
          f"{splits_df['n_images_used'].mean():.1f} ± {splits_df['n_images_used'].std(ddof=1):.1f}")

    # Save per-split details
    splits_df.to_csv(cfg["outfile"], index=False)
    print(f"Saved per-split stats to: {cfg['outfile']}")

    results[metric_name] = {"splits": splits_df, "summary": summary}


In [ ]:
# Load removed IDs
removed_ids = set()

if REMOVED_IMAGES_FILE.exists():
    print(f"Loading removed IDs from: {REMOVED_IMAGES_FILE}\n")

    with open(REMOVED_IMAGES_FILE, "r", encoding="utf-8") as f:
        for line in f:
            orig = line.rstrip("\n")
            line = orig.strip()
            if not line:
                continue

            file_path = Path(line)
            stem = file_path.stem       # e.g. "67" from "67.png"

            try:
                stem_int = int(stem)
            except ValueError:
                print(f"[WARN] Could not parse integer from stem '{stem}' (line: {orig})")
                continue

            removed_ids.add(stem_int)

    print("Final removed_ids set (ints):")
    print(removed_ids)
else:
    print(f"[WARN] removed_images file not found at {REMOVED_IMAGES_FILE}, no IDs will be excluded.")


# --- CONTAINERS (GLOBAL, ACROSS SUBJECTS) ---
quality = defaultdict(list)       # uniqueID (col B) -> [Q values]
authenticity = defaultdict(list)  # uniqueID (col B) -> [A values]
match = defaultdict(list)         # uniqueID (col B) -> [M values]

loaded_files = []
total_rows = 0

# --- PER-SUBJECT CONCENTRATION + RAW CORRELATIONS ---
per_subject_results = []  # one row per file/subject

# store per-subject histograms (20 bins each)
quality_histograms = []  # shape: (n_subjects, 20) after stacking
authenticity_histograms = []  # shape: (n_subjects, 20) after stacking


def _coerce_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def load_file(file_path: Path) -> pd.DataFrame:
    """
    Standardize to columns:
    ['uniqueID','Quality','Authenticity','Match']
    where uniqueID = column B; Q/A/M = columns E/F/G.
    Uses POSITIONAL columns so it's robust to header names.
    """
    suffix = file_path.suffix.lower()

    if suffix in {".xlsx", ".xls"}:
        # Read B..G then pick positions: B(0),E(3),F(4),G(5)
        dataframe = pd.read_excel(file_path, usecols="B:G", header=0)  # header names ignored below
        # If fewer than 6 columns were read, try without header
        if dataframe.shape[1] < 6:
            dataframe = pd.read_excel(file_path, usecols="B:G", header=None)
        # Ensure we have at least 6 cols (B..G)
        if dataframe.shape[1] < 6:
            raise ValueError(f"Expected at least 6 columns (B..G), got {dataframe.shape[1]}")

        unique_id_column = _coerce_numeric(dataframe.iloc[:, 0]).astype("Int64")  # column B
        quality_column = _coerce_numeric(dataframe.iloc[:, 3])                  # column E
        authenticity_column = _coerce_numeric(dataframe.iloc[:, 4])                  # column F
        match_column = _coerce_numeric(dataframe.iloc[:, 5])                  # column G

    elif suffix == ".csv":
        # Read all, then index by position safely
        try:
            raw_data = pd.read_csv(file_path, header=0)
        except Exception:
            raw_data = pd.read_csv(file_path, header=None)

        if raw_data.shape[1] < 7:
            raise ValueError(
                f"Expected at least 7 columns to access positions B,E,F,G; got {raw_data.shape[1]}"
            )

        # B is the second column (index 1) in your example
        unique_id_column = _coerce_numeric(raw_data.iloc[:, 1]).astype("Int64")  # column B (pos 1)
        quality_column = _coerce_numeric(raw_data.iloc[:, 4])                  # column E (pos 4)
        authenticity_column = _coerce_numeric(raw_data.iloc[:, 5])                  # column F (pos 5)
        match_column = _coerce_numeric(raw_data.iloc[:, 6])                  # column G (pos 6)

    else:
        raise ValueError(f"Unsupported file type: {file_path.name}")

    out = pd.DataFrame({
        "uniqueID": unique_id_column,
        "Quality": quality_column,
        "Authenticity": authenticity_column,
        "Match": match_column
    })

    # Drop rows with missing ID
    out = out.dropna(subset=["uniqueID"])
    # Drop rows with all-NaN metrics
    out = out.dropna(subset=["Quality", "Authenticity", "Match"], how="all")

    # Ensure uniqueID is Int64
    out["uniqueID"] = out["uniqueID"].astype("Int64")

    return out


# --- SELECT ONLY THE 25 SOURCE FILES ---
# they follow the pattern "##_scores.csv" (01_scores.csv ... 25_scores.csv):
source_csvs = sorted(SINGLE_SCORES_FILES.glob("*_scores.csv"))
source_excels = sorted(p for p in SINGLE_SCORES_FILES.iterdir() if p.suffix.lower() in {".xlsx", ".xls"})
# Prefer the explicit CSV pattern if that's your data; otherwise include Excels as well.
files = source_csvs if source_csvs else source_excels

if not files:
    raise FileNotFoundError(
        f"No source files found. Looked for '*_scores.csv' or Excel files in {SINGLE_SCORES_FILES}"
    )

# Define global binning for QA (0–5, 20 equal-width bins)
QUALITY_AUTHENTICITY_BINS = np.linspace(0, 5, 21)  # 20 bins


# --- LOAD ALL FILES ---
for file_path in files:
    try:
        dataframe = load_file(file_path)

        # --- FILTER OUT REMOVED IMAGES FOR *THIS* SUBJECT ---
        dataframe_subject = dataframe[~dataframe["uniqueID"].isin(removed_ids)].copy()

        # Update global per-image containers (unchanged from your original logic)
        for _, row in dataframe_subject.iterrows():
            uid = row["uniqueID"]  # Int64 / integer-like
            if pd.isna(uid):
                continue
            uid = int(uid)

            quality_val, authenticity_val, match_val = row["Quality"], row["Authenticity"], row["Match"]

            if pd.notna(quality_val):
                quality[uid].append(float(quality_val))
            if pd.notna(authenticity_val):
                authenticity[uid].append(float(authenticity_val))
            if pd.notna(match_val):
                match[uid].append(float(match_val))

        # --- PER-SUBJECT DATA ARRAYS ---
        quality_subject = pd.to_numeric(dataframe_subject["Quality"], errors="coerce").to_numpy(dtype=float)
        authenticity_subject = pd.to_numeric(dataframe_subject["Authenticity"], errors="coerce").to_numpy(dtype=float)

        # Drop NaNs
        mask_valid = ~np.isnan(quality_subject) & ~np.isnan(authenticity_subject)
        quality_subject = quality_subject[mask_valid]
        authenticity_subject = authenticity_subject[mask_valid]

        # --- INIT CORR VALUES ---
        pearson_bin_corr = np.nan
        spearman_bin_corr = np.nan
        pearson_raw_corr = np.nan
        spearman_raw_corr = np.nan

        # --- RAW CORRELATIONS (per subject) ---
        if quality_subject.size > 1 and authenticity_subject.size > 1:
            try:
                pearson_raw_corr, _ = pearsonr(quality_subject, authenticity_subject)
            except Exception:
                pearson_raw_corr = np.nan
            try:
                spearman_raw_corr, _ = spearmanr(quality_subject, authenticity_subject)
            except Exception:
                spearman_raw_corr = np.nan

        # --- BIN-CONCENTRATION CORRELATIONS (per subject) ---
        if quality_subject.size > 0 and authenticity_subject.size > 0:
            quality_counts, _ = np.histogram(quality_subject, bins=QUALITY_AUTHENTICITY_BINS)
            authenticity_counts, _ = np.histogram(authenticity_subject, bins=QUALITY_AUTHENTICITY_BINS)

            quality_total = quality_counts.sum()
            authenticity_total = authenticity_counts.sum()

            if quality_total > 0 and authenticity_total > 0:
                quality_probabilities = quality_counts / quality_total   # Quality concentration vector (20 bins)
                authenticity_probabilities = authenticity_counts / authenticity_total   # Authenticity concentration vector (20 bins)

                # store per-subject histograms
                quality_histograms.append(quality_probabilities)
                authenticity_histograms.append(authenticity_probabilities)

                try:
                    pearson_bin_corr, _ = pearsonr(quality_probabilities, authenticity_probabilities)
                except Exception:
                    pearson_bin_corr = np.nan
                try:
                    spearman_bin_corr, _ = spearmanr(quality_probabilities, authenticity_probabilities)
                except Exception:
                    spearman_bin_corr = np.nan

        per_subject_results.append({
            "subject_file": file_path.name,
            "n_QA": int(quality_subject.size),
            "pearson_bin_conc_QA": pearson_bin_corr,
            "spearman_bin_conc_QA": spearman_bin_corr,
            "pearson_raw_QA": pearson_raw_corr,
            "spearman_raw_QA": spearman_raw_corr,
        })

        loaded_files.append(file_path.name)
        total_rows += len(dataframe)

    except Exception as e:
        print(f"[WARN] Skipping {file_path.name}: {e}")


# --- OVERALL MEANS (UNCHANGED) ---
def flat_values(metric_dict):
    return [val for vals in metric_dict.values() for val in vals]


all_Q = np.array(flat_values(quality), dtype=float)
all_A = np.array(flat_values(authenticity), dtype=float)
all_M = np.array(flat_values(match), dtype=float)


def safe_mean(arr):
    return float(np.nanmean(arr)) if arr.size else float("nan")


overall_Q_mean = safe_mean(all_Q)
overall_A_mean = safe_mean(all_A)
overall_M_mean = safe_mean(all_M)

print("\n=== Overall Means ===")
print(f"Quality mean:       {overall_Q_mean:.4f} (N={all_Q.size})")
print(f"Authenticity mean:  {overall_A_mean:.4f} (N={all_A.size})")
print(f"Match mean:         {overall_M_mean:.4f} (N={all_M.size})")

# --- PER-IMAGE MEANS + COUNTS (UNCHANGED) ---
all_ids = sorted(set(quality) | set(authenticity) | set(match))
rows = []
for uid in all_ids:
    quality_vals = np.array(quality.get(uid, []), dtype=float)
    authenticity_vals = np.array(authenticity.get(uid, []), dtype=float)
    match_vals = np.array(match.get(uid, []), dtype=float)
    rows.append({
        "uniqueID": uid,
        "Q_mean": safe_mean(quality_vals), "Q_n": quality_vals.size,
        "A_mean": safe_mean(authenticity_vals), "A_n": authenticity_vals.size,
        "M_mean": safe_mean(match_vals), "M_n": match_vals.size,
    })

per_image_df = pd.DataFrame(rows).sort_values("uniqueID").reset_index(drop=True)

try:
    display(per_image_df.head(10))
except NameError:
    pass

BASE_DIR.mkdir(parents=True, exist_ok=True)
per_image_df.to_csv(SAVE_PER_IMAGE_CSV, index=False)
print(f"\nSaved per-image means to: {SAVE_PER_IMAGE_CSV}")

# --- SAVE PER-SUBJECT CORRELATIONS ---
per_subject_df = pd.DataFrame(per_subject_results)
BASE_DIR.mkdir(parents=True, exist_ok=True)
per_subject_df.to_csv(SAVE_PER_SUBJECT_CORR_CSV, index=False)

print(f"\nSaved per-subject correlations to: {SAVE_PER_SUBJECT_CORR_CSV}")
print(f"Loaded {len(loaded_files)} files (expected ~{EXPECT_N_FILES}).")
print(f"Total usable rows (before filtering by removed IDs): {total_rows}")
print(f"Unique images (after filtering): {len(all_ids)}")

try:
    display(per_subject_df)
except NameError:
    pass


# --- PLOT MEAN HISTOGRAMS ACROSS SUBJECTS (20 BINS, 0–5) ---
if quality_histograms and authenticity_histograms:
    quality_hist_arr = np.vstack(quality_histograms)  # (n_subjects, 20)
    authenticity_hist_arr = np.vstack(authenticity_histograms)  # (n_subjects, 20)

    quality_mean_hist = quality_hist_arr.mean(axis=0)
    authenticity_mean_hist = authenticity_hist_arr.mean(axis=0)

    bin_centers = 0.5 * (QUALITY_AUTHENTICITY_BINS[:-1] + QUALITY_AUTHENTICITY_BINS[1:])

    plt.figure(figsize=(6, 4))
    plt.plot(bin_centers, quality_mean_hist, marker="o", label="Quality")
    plt.plot(bin_centers, authenticity_mean_hist, marker="o", label="Authenticity")
    plt.xlabel("Rating (0–5)")
    plt.ylabel("Mean probability per bin")
    plt.title("Mean histograms across subjects (20 bins)")
    plt.legend()
    plt.tight_layout()
    fname = BASE_DIR / "mean_histograms_quality_authenticity.png"
    plt.savefig(fname, dpi=600, bbox_inches="tight")
    plt.show()

else:
    print("[INFO] No histograms collected; skipping mean histogram plot.")